In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [0]:
dbutils.widgets.text('incremental_flag','o')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')
print(incremental_flag)

In [0]:
df_src = spark.sql(
    """
    SELECT Branch_ID,BranchName FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
    """
)

In [0]:
df_src.display()

In [0]:
if not spark.catalog.tableExists('cars_catalog.gold.dim_branch'):
    df_sink = spark.sql(
                    """
                         SELECT 1 AS dim_branch_key,
                         Branch_ID,BranchName
                         FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
                        WHERE 1=0
                    """
    )
else:
    df_sink = spark.sql(
        '''
        SELECT dim_branch_key,Branch_ID,BranchName
        FROM cars_catalog.gold.dim_branch
        '''
    )

In [0]:
df_sink.display()

In [0]:
df_filter = df_src.join(df_sink, df_src.Branch_ID == df_sink.Branch_ID,'left').select(df_src.Branch_ID,df_src.BranchName,df_sink.dim_branch_key)
df_filter = df_filter.dropDuplicates(["dim_branch_key"])
df_filter.display()

In [0]:
df_filter_old = df_filter.filter(df_filter.dim_branch_key.isNotNull())
df_filter_old.display()

In [0]:
df_filter_new = df_filter.filter(df_filter.dim_branch_key.isNull()).select("Branch_ID","BranchName")
df_filter_new.display()

In [0]:
if (incremental_flag == '0'):
    max_value=0
else:
    if spark.catalog.tableExists('cars_catalog.gold.dim_branch'):
        max_value = spark.sql("SELECT MAX(dim_branch_key) FROM cars_catalog.gold.dim_branch").first()[0]

if spark.catalog.tableExists('cars_catalog.gold.dim_branch'):
    max_value = spark.sql("SELECT MAX(dim_branch_key) FROM cars_catalog.gold.dim_branch").first()[0]
else:
    max_value = 0
print(max_value)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

w = Window.orderBy(F.lit(1))
df_filter_new = df_filter_new.withColumn('dim_branch_key',
                                         F.lit(max_value)+F.row_number().over(w))
df_filter_new.display()
                                        

In [0]:
df_final = df_filter_new.union(df_filter_old)
df_final.display()

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists('cars_catalog.gold.dim_branch'):
    delta_table = DeltaTable.forPath(spark,"abfss://gold@carsalesdata.dfs.core.windows.net/dim_branch")
    delta_table.alias("target").merge(df_final.alias("source"),"target.dim_branch_key = source.dim_branch_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@carsalesdata.dfs.core.windows.net/dim_branch")\
        .saveAsTable("cars_catalog.gold.dim_branch", overwrite=True)

In [0]:
%sql
select * from cars_catalog.gold.dim_branch